<a href="https://colab.research.google.com/drive/1ForCMIoCgiM7jtrWw06xSCzm70p1FZZI" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TopicScope
----

Please run all the cells in this notebook in order.
Full functionality will be released after the completion of the patent application. At this time, we have only confirmed that it works on the T4 and A100 GPU.

# Environment Setup

In [ ]:
!sudo apt update -qq > environment_setup.log 2>&1
!sudo apt install pciutils lshw -qq >> environment_setup.log 2>&1
!sudo apt install zstd -qq >> environment_setup.log 2>&1
!curl -fsSL https://ollama.com/install.sh | sh >> environment_setup.log 2>&1
!apt-get -y install fonts-ipafont-gothic -qq >> environment_setup.log 2>&1
!echo 'done.'

print('Please wait in about 10m for downloading.')

import threading
import subprocess
import time

def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

threading.Thread(target=run_ollama_serve).start()
time.sleep(5)
!echo 'ollama service start.'

!pip install langchain-ollama > ollama_setup.log 2>&1

# If you want to try other models, just change this.
!ollama pull llama3.2 >> ollama_setup.log 2>&1
!ollama pull gpt-oss:20b >> ollama_setup.log 2>&1

!echo "FROM gpt-oss:20b" > Modelfile
!echo "PARAMETER num_ctx 65536" >> Modelfile
!ollama create gpt-oss:20b-64k -f Modelfile >> ollama_setup.log 2>&1

!ollama list
!echo 'models are listed up.'

done.
ollama service start.
NAME               ID              SIZE      MODIFIED               
gpt-oss:20b        17052f91a42e    13 GB     Less than a second ago    
gpt-oss:20b-64k    d3283273b3f6    13 GB     Less than a second ago    
llama3.2:latest    a80c4f17acd5    2.0 GB    2 minutes ago             
models are listed up.


In [ ]:
# Load libraries
!pip -q install gdown ipywidgets

import gdown
import subprocess
import sys
from IPython.display import display
import ipywidgets as widgets

# UI
file_id_input = widgets.Text(
    value='',
    placeholder='Author\'s last name',
    description='FILE_ID:',
    layout=widgets.Layout(width='300px')
)

run_button = widgets.Button(
    description='Press the button',
    button_style='success'
)

output_area = widgets.Output()

# Process after the button is pressed.
def on_run_clicked(b):
    with output_area:
        output_area.clear_output()

        file_id = file_id_input.value.strip()
        if not file_id:
            print("Please input FILE_ID")
            return

        print(f"Downloading FILE_ID: {file_id}")

        url = f"https://drive.google.com/uc?id=1PS{file_id[1]}ZSkOO{file_id[3]}0g{file_id[5]}kFVEgXamuY_RmbT9-{file_id[4]}-o"
        output_file = "topicscope.zip"

        try:
            gdown.download(url, output_file, quiet=False)
        except Exception as e:
            print("Fail to download:", e)
            return

        print("\n--- START ---\n")
        !unzip {output_file}
        %run topicscope.py
        print("\n--- FINISH ---\n")

# Resister process
run_button.on_click(on_run_clicked)



# Download Programs

In [ ]:
print('Please enter ' + '\033[31m' + 'the author\'s last name in lowercase' + '\033[0m' + ' and press the green button. This process will take at least 10 minutes.\n')

# Display
display(file_id_input, run_button, output_area)

# Summarization Demo

In [ ]:
from google.colab import output
output.no_vertical_scroll()
from IPython.display import Javascript
display(Javascript('google.colab.output.setIframeHeight(0, true{maxHeight: 640})'))

#@title TopicScope

# @markdown Please specify the number of topics (<=16) before executing the cell.

topic_number = 12 #@param {type:"number"}

retriever = None

from IPython.display import display, HTML, Javascript
import base64

html_code = f"""
<div style="font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; max-width: 980px;">
  <h2 style="margin: 0 0 8px 0;">Interactive Large-Document Summarization (Local LLMs)</h2>
  <div style="color:#444; margin-bottom: 12px;">
    Visualizing <b>Topic Preservation</b> and <b>Latency</b> trade-offs (K={topic_number}).
    After uploading the English text file, when the Word Cloud appears, select a method and press Run (Summary).
  </div>

  <!-- File Upload -->
  <div style="display:flex; gap:12px; align-items:center; margin: 10px 0 14px;">
    <label style="min-width:130px;">File upload:</label>
    <input type="file" id="file-input" />
    <div id="file-name"></div>
  </div>

  <!-- Controls -->
  <div style="display:flex; gap:12px; align-items:center; margin: 10px 0 14px;">
    <label style="min-width:130px;">Method:</label>
    <select id="methodSel" style="flex:1; padding:6px;">
      <option value="hierarchical">Hierarchical</option>
      <option value="topicbased">Topic-based</option>
      <option value="topicbasedrag">Topic-based RAG</option>
      <option value="topicdrivenrag">Topic-driven RAG</option>
    </select>
    <button id="runBtn" style="padding:7px 12px; cursor:pointer;">Run (Summary)</button>
  </div>
</div>

<script>
document.getElementById("runBtn").addEventListener("click", ()=>{{
  const method = document.getElementById("methodSel").value;
  google.colab.kernel.invokeFunction('notebook.summarize', [method], {{}});
}});

document.getElementById('file-input').addEventListener('change', function(e) {{
    var file = e.target.files[0];
    if (!file) return;

    // Display filename
    document.getElementById('file-name').innerText = 'Selected: ' + file.name;

    // Process for loading files in JavaScript and passing them to Python in Colab
    var reader = new FileReader();
    reader.onload = function(e) {{
        var contents = e.target.result;
        // Convert to base64 format and pass it to the Python side
        var base64 = contents.split(',')[1];
        // Passing data to Python variables through Google objects in collaboration
        google.colab.kernel.invokeFunction('notebook.upload_file', [base64, file.name], {{}});
    }};
    reader.readAsDataURL(file);
}});

/** ====== Data (paste your averages here) ======
 * You can keep exactly these numbers (from your table), or replace later.
 * If you want per-document results, store them under docs.doc1 / doc2 / doc3.
 */
const methods = [
  {{
    key: "hierarchical",
    name: "Hierarchical Summarization",
    latency: 0.0,
    cov: 0.0,
    covJS: 0.0,
    summary: {{
      doc1: "Preview: hierarchical long-doc summary (precomputed).",
      doc2: "Preview: hierarchical long-doc summary (precomputed).",
      doc3: "Preview: hierarchical long-doc summary (precomputed).",
      doc4: "Preview: hierarchical long-doc summary (precomputed).",
      doc5: "Preview: hierarchical long-doc summary (precomputed).",
    }}
  }},
  {{
    key: "topic_based",
    name: "Topic-based Summarization",
    latency: 0.0,
    cov: 0.0,
    covJS: 0.0,
    summary: {{
      doc1: "Preview: hierarchical long-doc summary (precomputed).",
      doc2: "Preview: hierarchical long-doc summary (precomputed).",
      doc3: "Preview: hierarchical long-doc summary (precomputed).",
      doc4: "Preview: hierarchical long-doc summary (precomputed).",
      doc5: "Preview: hierarchical long-doc summary (precomputed).",
    }}
  }},
  {{
    key: "topic_based_rag",
    name: "Topic-based RAG Summarization",
    latency: 0.0,
    cov: 0.0,
    covJS: 0.0,
    summary: {{
      doc1: "Preview: hierarchical long-doc summary (precomputed).",
      doc2: "Preview: hierarchical long-doc summary (precomputed).",
      doc3: "Preview: hierarchical long-doc summary (precomputed).",
      doc4: "Preview: hierarchical long-doc summary (precomputed).",
      doc5: "Preview: hierarchical long-doc summary (precomputed).",
    }}
  }},
  {{
    key: "topic_driven_rag",
    name: "Topic-driven RAG Summarization",
    latency: 0.0,
    cov: 0.0,
    covJS: 0.0,
    summary: {{
      doc1: "Preview: hierarchical long-doc summary (precomputed).",
      doc2: "Preview: hierarchical long-doc summary (precomputed).",
      doc3: "Preview: hierarchical long-doc summary (precomputed).",
      doc4: "Preview: hierarchical long-doc summary (precomputed).",
      doc5: "Preview: hierarchical long-doc summary (precomputed).",
    }}
  }},
];

function fmt(x, d=3) {{ return (Math.round(x * 10**d) / 10**d).toFixed(d); }}
function fmtS(x) {{ return Math.round(x).toString(); }}

function renderTable(docId) {{
  const tbody = document.querySelector("#tbl tbody");
  tbody.innerHTML = "";
  for(const m of methods) {{
    const tr = document.createElement("tr");
    tr.style.borderBottom = "1px solid #f0f0f0";
    tr.innerHTML = `
      <td style="padding:8px; font-weight:600;">${{m.name}}</td>
      <td style="padding:8px;">${{fmtS(m.latency,1)}}</td>
      <td style="padding:8px;">${{fmt(m.cov,3)}}</td>
      <!--<td style="padding:8px;">${{fmt(m.covJS,3)}}</td>-->
      <td style="padding:8px; color:#333;">${{m.summary[docId] || ""}}</td>
    `;
    tbody.appendChild(tr);
  }}
}}

function escapeHTML(str) {{
  if (typeof str !== 'string') return str;
  const escape = {{
    '&': '&amp;',
    '<': '&lt;',
    '>': '&gt;',
    '"': '&quot;',
    "'": '&#39;',
  }};
  return str.replace(/[&<>"']/g, function (match) {{
    return escape[match];
  }});
}}
</script>
"""

result_code = f"""
<div style="font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; max-width: 980px;">
  <!-- Controls -->
  <!--
  <div style="display:flex; gap:12px; align-items:center; margin: 10px 0 14px;">
    <label style="min-width:130px;">Sample document:</label>
    <select id="docSel" style="flex:1; padding:6px;">
      <option value="all">All Topics</option>
    </select>
    <button id="runBtn" style="padding:7px 12px; cursor:pointer;">Run (Summary)</button>
  </div>
  -->

  <!-- Comparison table -->
  <div style="border:1px solid #ddd; border-radius:10px; padding:12px; margin-bottom:14px;">
    <div style="font-weight:700; margin-bottom:8px;">4-method comparison</div>
    <table id="tbl" style="border-collapse: collapse; width:100%; font-size: 14px;">
      <thead>
        <tr style="text-align:left; border-bottom:1px solid #ddd;">
          <th style="padding:8px;">Method</th>
          <th style="padding:8px;">Latency (s)</th>
          <th style="padding:8px;">Topic Coverage</th>
          <th style="padding:8px;">Summary (head)</th>
        </tr>
      </thead>
      <tbody></tbody>
    </table>
  </div>

  <!-- Summary erea -->
  <div style="border:1px solid #ddd; border-radius:10px; padding:12px; margin-bottom:14px;">
    <div style="font-weight:700; margin-bottom:8px;">Summary area</div>
    <textarea id="tarea" style="height: 300px; width: 100%;" name="textarea"></textarea>
  </div>

</div>
"""

# --- Function to receive files on the Python side ---
from google.colab import output

summary_h_ = ""
summary_t_ = ""
summary_b_ = ""
summary_d_ = ""
indexing_time = 0
processing_time_h = 0
processing_time_t = 0
processing_time_b = 0
processing_time_d = 0
cov_b_h = 0
cov_b_t = 0
cov_b_b = 0
cov_b_d = 0

def summarize(method):
  global retriever
  global summary_h_
  global summary_t_
  global summary_b_
  global summary_d_
  global indexing_time
  global processing_time_h
  global processing_time_t
  global processing_time_b
  global processing_time_d
  global cov_b_h
  global cov_b_t
  global cov_b_b
  global cov_b_d

  if (method == 'hierarchical'):
    js_code = f"""speak("I have commenced summarising using a Hierarchical approach.");"""
    display(Javascript(js_code))

    summary_h, processing_time_h = hierarchical(texts)
    summary_h_ = summary_h.data[:300].replace('\n', ' ').replace('\r', '')
    summary_h_all = summary_h.data[:].replace('\n', ' ').replace('\r', '')
    cov_b_h, cov_f_h = topic_coverage(summary_h.data, topic_words)
    js_code = f"""
      const selectElement = document.getElementById('tarea');
      selectElement.value = escapeHTML('{summary_h_all}');
    """
    display(Javascript(js_code))

    js_code = f"""speak("The Hierarchical summary has been completed.");"""
    display(Javascript(js_code))

  if (method == 'topicbased'):
    js_code = f"""speak("I have commenced summarising using a Topic-based approach.");"""
    display(Javascript(js_code))

    retriever, indexing_time = rag_indexing(texts)
    summary_t, processing_time_t = topic_based(top_k_words)
    summary_t_ = summary_t.data[:300].replace('\n', ' ').replace('\r', '')
    summary_t_all = summary_t.data[:].replace('\n', ' ').replace('\r', '')
    cov_b_t, cov_f_t = topic_coverage(summary_t.data, topic_words)
    js_code = f"""
      const selectElement = document.getElementById('tarea');
      selectElement.value = escapeHTML('{summary_t_all}');
    """
    display(Javascript(js_code))

    js_code = f"""speak("The Topic-based summary has been completed.");"""
    display(Javascript(js_code))

  if (method == 'topicbasedrag'):
    js_code = f"""speak("I have commenced summarising using a Topic-based RAG approach.");"""
    display(Javascript(js_code))

    retriever, indexing_time = rag_indexing(texts)
    summary_b, processing_time_b = topic_based_rag(top_k_words)
    summary_b_ = summary_b.data[:300].replace('\n', ' ').replace('\r', '')
    summary_b_all = summary_b.data[:].replace('\n', ' ').replace('\r', '')
    cov_b_b, cov_f_b = topic_coverage(summary_b.data, topic_words)
    js_code = f"""
      const selectElement = document.getElementById('tarea');
      selectElement.value = escapeHTML('{summary_b_all}');
    """
    display(Javascript(js_code))

    js_code = f"""speak("The Topic-based RAG summary has been completed.");"""
    display(Javascript(js_code))

  if (method == 'topicdrivenrag'):
    js_code = f"""speak("I have commenced summarising using a Topic-driven RAG approach.");"""
    display(Javascript(js_code))

    retriever, indexing_time = rag_indexing(texts)
    summary_d, processing_time_d = topic_based_rag(top_k_words)
    summary_d_ = summary_d.data[:300].replace('\n', ' ').replace('\r', '')
    summary_d_all = summary_d.data[:].replace('\n', ' ').replace('\r', '')
    cov_b_d, cov_f_d = topic_coverage(summary_d.data, topic_words)
    js_code = f"""
      const selectElement = document.getElementById('tarea');
      selectElement.value = escapeHTML('{summary_d_all}');
    """
    display(Javascript(js_code))

    js_code = f"""speak("The Topic-driven RAG summary has been completed.");"""
    display(Javascript(js_code))

  processing_time_b_ = processing_time_b + indexing_time
  processing_time_d_ = processing_time_d + indexing_time
  js_code = f"""
    docId = 'doc1';
    for (const m of methods) {{
      if (m.key == 'hierarchical') {{
          m.summary[docId] = escapeHTML('{summary_h_}');
          m.latency = {processing_time_h};
          m.cov = {cov_b_h};
      }}
      else if (m.key == 'topic_based') {{
          m.summary[docId] = escapeHTML('{summary_t_}');
          m.latency = {processing_time_t};
          m.cov = {cov_b_t};
      }}
      else if (m.key == 'topic_based_rag') {{
          m.summary[docId] = escapeHTML('{summary_b_}');
          m.latency = {processing_time_b_};
          m.cov = {cov_b_b};
      }}
      else if (m.key == 'topic_driven_rag') {{
          m.summary[docId] = escapeHTML('{summary_d_}');
          m.latency = {processing_time_d_};
          m.cov = {cov_b_d};
      }}
    }}
    renderTable(docId);
  """;
  display(Javascript(js_code))

file_name = ''
topic_words = []

def upload_file(base64_str, fname):
  global file_name
  global topic_words
  file_name = fname

  # Decode the base64 and save it.
  #print('uploading...', end="")
  #start = time.perf_counter() # start
  with open(file_name, 'wb') as f:
    f.write(base64.b64decode(base64_str))
  #end = time.perf_counter() # finish
  #print('done. {:.2f} s'.format((end - start)))
  print(f'File "{file_name}" uploaded successfully.')

  js_code = f"""speak("Upload complete. We will now proceed with topic modeling. Once topic modeling is complete, select Method to generate a summary.");"""
  display(Javascript(js_code))

  # Parse the uploaded file.
  texts = []
  start = time.perf_counter() # start
  with open(file_name) as f:
    for line in f:
      texts.append(line.rstrip('\n'))
  components, top_k_words = parse(texts, topic_number)
  topic_words = sum([list(words) for words in top_k_words], [])
  #print(f'File "{file_name}" parsed.')
  print('Topic modelling has been completed. Please select a method to summarise.')

  js_code = """
    const selectElement = document.getElementById('docSel');
    selectElement.options.length = 0;
    let option = document.createElement("option");
    option.value = "all";
    option.text = "All Topics";
    selectElement.appendChild(option);
  """;
  for i, comp in enumerate(components):
    js_code += f"""
    option = document.createElement("option");
    option.value = "doc{i}";
    option.text = "Topic #{i}";
    selectElement.appendChild(option);
    """;
  #display(HTML(js_code))
  display(HTML(result_code))

  js_code = f"""speak("Topic modelling has been completed. Please select a method to summarise.");"""
  display(Javascript(js_code))

# Register function calls from JavaScript
output.register_callback('notebook.upload_file', upload_file)
output.register_callback('notebook.summarize', summarize)

# Display HTML
display(HTML(html_code))

voice_code = f"""
function speak(text) {{
  const utterance = new SpeechSynthesisUtterance(text);
  utterance.lang = 'en-US';
  utterance.rate = 1.0;
  utterance.pitch = 1.0;

  function selectVoice() {{
    const voices = speechSynthesis.getVoices();
    if (!voices.length) return;
    let voice =
      voices.find(v =>
        v.lang.startsWith('en') &&
        /(Kyoko|Haruka|Ayumi|Otoya|Japan|Microsoft)/i.test(v.name)
      );
    if (!voice) {{
      voice = voices.find(v =>
        v.lang.startsWith('en') &&
        /(female|zira|samantha|aria|jenny|emily)/i.test(v.name)
      );
    }}
    if (!voice) {{
      voice = voices.find(v => v.lang.startsWith('en'));
    }}
    if (voice) {{
      utterance.voice = voice;
    }}
    speechSynthesis.speak(utterance);
  }}

  if (speechSynthesis.getVoices().length) {{
    selectVoice();
  }} else {{
    speechSynthesis.onvoiceschanged = selectVoice;
  }}
}}
"""
display(Javascript(voice_code))

# Speak
js_code = f"""speak("The number of topics is set to {topic_number}. Upload the file and let's summarize it.");"""
display(Javascript(js_code))